# Glass Quality Prediction  Weekend Hackathon 6

We humans have been using glass since ancient times for a variety of applications from building construction to making decorative objects. With technology, glass and its applications have evolved, and today, we have different varieties of glass used for very different purposes from a computer monitor to a bulletproof car window depending on the grade of the glass produced. And not all grades or varieties are manufactured the same way. In this data science challenge, you as a data scientist must use the given data to predict the grade of the glass produced based on the given factors.

**Given are 15 distinguishing factors that can provide insight into what grade of the glass is being produced. Your objective as a data scientist is to build a machine learning model that can predict the grade of glass based on the given factors.**

**Data Description:-**
The unzipped folder will have the following files.

- Train.csv – 1358 observations.
- Test.csv – 583 observations.
- Sample Submission – Sample format for the submission.
- Target Variable: class

## Import Modules

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.ensemble import ExtraTreesClassifier

## Import Dataset

In [ ]:
train = pd.read_csv("data/Train.csv")
test = pd.read_csv("data/Test.csv")

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
train[['grade_A_Component_1','grade_A_Component_2','x_component_1','x_component_2','x_component_3','x_component_4','x_component_5']].head() 

**Note:** Grade_A feature and x_component are one hot encoded.
- grade_A_Component_1 and grade_A_component_2 represent the same, there is a dummy variable.
- So, **grade_A_Component_1** is removed from features.
- In case of x_component, that is not the case all the 5 features are needed.

In [ ]:
train[['pixel_area','log_area']].head()

**Note**: Since the interger part of pixel_area and log_area are same, we remove pixel_area or log_area by considering the feature importance and log_losss metric.

## Features

These are features which are created

In [ ]:
def create_features(df):
    
    df['x'] = df['xmax']-df['xmin']
    df['y'] = df['ymax']-df['ymin']
    df['area'] = df['x']*abs(df['y'])
    df['volume'] = df['area']*df['thickness']
    df['diff_area'] = df['log_area'] - df['pixel_area']
    
    return df

In [ ]:
train = create_features(train)
test = create_features(test)

## EDA

In [ ]:
# Plot the distribution plots for the numeric features
fig, ax = plt.subplots(nrows=6, ncols=2,figsize=(12,12))

ax = ax.flatten()
i=0

for feat in ['max_luminosity', 'thickness','xmin','xmax','ymin', 'ymax','pixel_area','log_area','x','y','area']:
    sns.histplot(data=train, x=feat, bins=20, kde=True, ax=ax[i])
    i+=1
    
fig.delaxes(ax[11])
fig.tight_layout()

In [ ]:
# Plot the distribution plots for the numeric features
fig, ax = plt.subplots(nrows=3, ncols=2,figsize=(12,12))

ax = ax.flatten()
i=0

for feat in ['x_component_1','x_component_2','x_component_3','x_component_4','x_component_5']:
    sns.countplot(x=feat,data=train,hue='class',ax=ax[i])
    i+=1
    
fig.delaxes(ax[5])
fig.tight_layout()

**Note:** From these plots we can see that x_component with 1's are all classified as class 1.
- This can be used directly for classification with probabilty as 1 for class 1.

# features

After trying models with different features these features provided lower log_loss

In [ ]:
features = [
    column for column in train.columns
    if column not in {'class', 'grade_A_Component_1', 'y', 'pixel_area', 'diff_area'}
]
target = 'class'
features

In [ ]:
def metric(y, y0, labels):
    return log_loss(y, y0, labels=labels)

In [ ]:
def k_fold_cross_valid(model, train, features, target, n_splits=5):
    X = train[features].copy()
    y = train[target].copy()
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=11)
    scores = []
    for train_index, test_index in splitter.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        model.fit(X_train, y_train)
        y_pred = model.predict_proba(X_test)
        scores.append(metric(y_test, y_pred, model.classes_))
    print("LogLoss:", np.mean(scores))

In [ ]:
model = ExtraTreesClassifier(random_state=11)
k_fold_cross_valid(model,train,features,target,n_splits=5)

In [ ]:
model = ExtraTreesClassifier(random_state=11)
model.fit(train[features], train[target])
y_pred = model.predict_proba(test[features])
sub = pd.DataFrame(y_pred, columns=model.classes_)

In [ ]:
sub.to_excel("extratrees_addv1.xlsx", index=False)

In [ ]:
sub.head()